<a href="https://colab.research.google.com/github/yhshengjy/ClinPKPD/blob/main/Notebook7_aminoglycoside_PKPD_simulation_english.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 7: Aminoglycoside PK/PD Simulation

This notebook is one of the antibacterial drug modules in the clinical pharmacy PK/PD interactive simulation platform.

This section focuses on **aminoglycosides** as the teaching topic and uses **gentamicin** as the example drug to help you understand the PK/PD characteristics of concentration-dependent antibacterial agents.

The key teaching points for aminoglycosides include:

- Concentration-dependent killing
- Post-antibiotic effect, PAE
- The relationship between peak concentration and MIC
- The relationship between trough concentration and toxicity risk
- Differences between once-daily dosing and traditional divided dosing
- The impact of reduced renal function on clearance, half-life, and accumulation

The core logic of this notebook is:

$$
Dose \rightarrow C(t) \rightarrow C_{max}/MIC \rightarrow Efficacy
$$

At the same time, we also need to consider:

$$
C_{min} \rightarrow Accumulation \rightarrow Toxicity\ Risk
$$

**This notebook is intended for teaching simulation only and should not be used for real patient prescribing or dose calculation.** Real clinical use requires consideration of the product label, institutional antimicrobial stewardship protocols, TDM results, infection site, pathogen MIC, changes in renal function, and clinical judgment by physicians and pharmacists.


## 1. Learning Objectives

After completing this notebook, you should be able to:

1. Explain why aminoglycosides are considered concentration-dependent antibiotics.
2. Interpret Cmax/MIC as a key PK/PD index for aminoglycoside efficacy.
3. Compare divided dosing and extended-interval dosing using peak and trough concentrations.
4. Describe how renal function affects clearance, half-life, accumulation, and toxicity risk.
5. Use a one-compartment infusion model to evaluate preliminary PK/PD target attainment.


## 2. PK/PD Characteristics of Aminoglycosides

Aminoglycoside antibiotics are commonly used to treat severe Gram-negative bacterial infections. Common agents include:

- Gentamicin
- Tobramycin
- Amikacin

Aminoglycosides have the following PK/PD characteristics:

### 2.1 Concentration-Dependent Killing

The antibacterial effect of aminoglycosides is usually related to peak concentration. In general, higher concentrations are associated with faster and stronger bacterial killing.

Therefore, a commonly used PK/PD index for aminoglycosides is:

$$
\frac{C_{max}}{MIC}
$$

where:

| Symbol | Meaning |
|---|---|
| Cmax | Peak concentration |
| MIC | Minimum inhibitory concentration |
| Cmax/MIC | Ratio of peak concentration to MIC |

A commonly used empirical teaching target is:

$$
C_{max}/MIC \geq 8\text{--}10
$$

This means that when the peak concentration is approximately 8 to 10 times the MIC, a better antibacterial effect is usually more likely. Note that this is only a teaching target. The appropriate target may differ across drugs, infection sites, pathogens, and patient conditions.

### 2.2 Post-Antibiotic Effect, PAE

Aminoglycosides have a post-antibiotic effect. Even after the plasma concentration falls below the MIC, bacterial growth may remain suppressed for a period of time.

This is one of the important theoretical reasons why extended dosing intervals or once-daily dosing may be considered in certain situations.

### 2.3 Trough Concentration and Toxicity Risk

Aminoglycosides carry risks of nephrotoxicity and ototoxicity. Unlike efficacy, which is more closely related to peak concentration, safety often requires attention to trough concentration and drug accumulation.

In teaching simulations, we can use:

$$
C_{min}
$$

to represent the trough concentration at the end of a dosing interval.

If the trough concentration remains high over time, this suggests insufficient drug clearance or increased accumulation risk.


## 3. Rationale for Using Gentamicin as the Teaching Example

This notebook uses **gentamicin** as the example drug.

Reasons for this choice:

- Gentamicin is a classic aminoglycoside.
- It is mainly eliminated by the kidneys, making it suitable for demonstrating the impact of renal function changes on PK.
- The official product label clearly recommends monitoring peak and trough concentrations to ensure efficacy and avoid excessive concentrations.
- The label states that clearance is reduced in patients with renal impairment and that dose adjustment is required.

Gentamicin can be administered by IM or IV routes. For intermittent IV administration, it is commonly infused over 0.5 to 2 hours. Gentamicin clearance is reduced in patients with impaired renal function; the more severe the renal impairment, the slower the clearance, so dose adjustment is required.

**The parameters used in this notebook are simplified teaching values and do not represent the true parameters of any specific patient.**


## 4. One-Compartment Multiple-Dose Intravenous Infusion Model

Aminoglycosides are often administered clinically by intravenous infusion. This notebook uses a one-compartment intermittent intravenous infusion model.

For a single intravenous infusion:

- The administered dose is Dose
- The infusion duration is Tinf
- The infusion rate is:

$$
R_0 = \frac{Dose}{T_{inf}}
$$

The elimination rate constant is:

$$
k = \frac{CL}{V_d}
$$

During the infusion, concentration rises:

$$
C(t) = \frac{R_0}{CL}\left(1-e^{-kt}\right)
$$

After the infusion ends, concentration declines:

$$
C(t) = \frac{R_0}{CL}\left(1-e^{-kT_{inf}}\right)e^{-k(t-T_{inf})}
$$

With multiple dosing, the concentration-time curves generated by each dose can be added together. This is the principle of **superposition**.

The multiple-dose model used in this notebook is based on this idea.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True


def gentamicin_clearance_from_crcl(crcl_ml_min):
    """
    Approximate gentamicin clearance from creatinine clearance.

    This simplified teaching model assumes gentamicin clearance is
    proportional to creatinine clearance.

    1 mL/min = 0.06 L/h
    """
    cl_l_h = crcl_ml_min * 0.06
    return cl_l_h


def one_compartment_iv_infusion_multiple(
    t,
    dose_mg,
    vd_l,
    cl_l_h,
    tau_h,
    infusion_h,
    n_doses
):
    """
    One-compartment multiple-dose IV infusion model.
    """
    k_elim = cl_l_h / vd_l
    rate_mg_h = dose_mg / infusion_h
    concentration = np.zeros_like(t, dtype=float)

    for dose_number in range(n_doses):
        dose_time = dose_number * tau_h
        elapsed = t - dose_time

        during_infusion = (elapsed >= 0) & (elapsed <= infusion_h)
        after_infusion = elapsed > infusion_h

        concentration[during_infusion] += (
            rate_mg_h / cl_l_h
            * (1 - np.exp(-k_elim * elapsed[during_infusion]))
        )

        concentration[after_infusion] += (
            rate_mg_h / cl_l_h
            * (1 - np.exp(-k_elim * infusion_h))
            * np.exp(-k_elim * (elapsed[after_infusion] - infusion_h))
        )

    half_life_h = np.log(2) / k_elim
    return concentration, k_elim, half_life_h


def calculate_interval_metrics(t, concentration, tau_h, infusion_h, n_doses):
    """
    Calculate approximate last-interval Cmax and Cmin.
    """
    last_dose_time = (n_doses - 1) * tau_h
    interval_mask = (t >= last_dose_time) & (t <= last_dose_time + tau_h)

    t_interval = t[interval_mask]
    c_interval = concentration[interval_mask]

    cmax = np.max(c_interval)
    tmax = t_interval[np.argmax(c_interval)] - last_dose_time

    cmin = c_interval[-1]
    t_cmin = t_interval[-1] - last_dose_time

    auc_interval = np.trapz(c_interval, t_interval)
    auc24 = auc_interval * (24 / tau_h)

    return {
        "Cmax_last_interval": cmax,
        "Tmax_after_dose": tmax,
        "Cmin_last_interval": cmin,
        "Tmin_after_dose": t_cmin,
        "AUC_interval": auc_interval,
        "AUC24": auc24
    }


def aminoglycoside_pd_effect(concentration, mic, emax=100, ec50_ratio=4, gamma=2):
    """
    Simplified PD effect model based on concentration/MIC ratio.
    This is for teaching only.
    """
    ratio = concentration / mic
    effect = emax * (ratio ** gamma) / (ec50_ratio ** gamma + ratio ** gamma)
    return effect


def status_label(value, threshold, direction="above"):
    if direction == "above":
        return "Target attained" if value >= threshold else "Below target"
    if direction == "below":
        return "Acceptable" if value <= threshold else "High risk"
    return ""

## 5. Interactive Simulation 1: Multiple-Dose Gentamicin Concentration-Time Profile

The simulation below shows the multiple-dose concentration-time profile after intermittent intravenous infusion of gentamicin.

You can adjust:

- Body weight
- Dose, calculated in mg/kg
- Dosing interval, tau
- Infusion time
- Renal function, CrCl
- Volume of distribution, Vd
- MIC

Focus on the following questions:

- Is Cmax high enough?
- Does Cmax/MIC reach the teaching target?
- Is Cmin too high?
- Does accumulation occur when renal function declines?


In [ ]:
def plot_aminoglycoside_multiple_dosing(
    body_weight_kg=70,
    dose_mg_kg=5.0,
    tau_h=24,
    infusion_h=1.0,
    n_doses=5,
    vd_l_kg=0.25,
    crcl_ml_min=100,
    mic_mg_l=1.0,
    target_cmax_mic=8,
    trough_threshold=2.0
):
    dose_mg = body_weight_kg * dose_mg_kg
    vd_l = body_weight_kg * vd_l_kg
    cl_l_h = gentamicin_clearance_from_crcl(crcl_ml_min)

    t_end_h = tau_h * n_doses
    t = np.linspace(0, t_end_h, 3000)

    concentration, k_elim, half_life_h = one_compartment_iv_infusion_multiple(
        t=t,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        tau_h=tau_h,
        infusion_h=infusion_h,
        n_doses=n_doses
    )

    metrics = calculate_interval_metrics(
        t=t,
        concentration=concentration,
        tau_h=tau_h,
        infusion_h=infusion_h,
        n_doses=n_doses
    )

    cmax_mic = metrics["Cmax_last_interval"] / mic_mg_l
    effect = aminoglycoside_pd_effect(concentration, mic_mg_l)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.plot(t, concentration, linewidth=2, label="Gentamicin concentration")
    ax1.axhline(mic_mg_l, linestyle="--", label=f"MIC = {mic_mg_l:.2f} mg/L")
    ax1.axhline(trough_threshold, linestyle=":", label=f"Trough threshold = {trough_threshold:.1f} mg/L")
    ax1.set_xlabel("Time (h)")
    ax1.set_ylabel("Concentration (mg/L)")

    ax2 = ax1.twinx()
    ax2.plot(t, effect, linestyle="-.", linewidth=2, label="PD effect score")
    ax2.set_ylabel("Effect score (%)")
    ax2.set_ylim(0, 105)

    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right")
    ax1.set_title("Aminoglycoside Multiple-Dose PK/PD Simulation")
    plt.show()

    summary = pd.DataFrame({
        "Metric": [
            "Dose per administration",
            "Total daily dose",
            "Vd",
            "CL",
            "Elimination rate constant",
            "Half-life",
            "Last-interval Cmax",
            "Time of Cmax after dose",
            "Last-interval Cmin",
            "AUC24",
            "MIC",
            "Cmax/MIC",
            "Efficacy target",
            "Trough risk"
        ],
        "Value": [
            f"{dose_mg:.1f} mg",
            f"{dose_mg * 24 / tau_h:.1f} mg/day",
            f"{vd_l:.1f} L",
            f"{cl_l_h:.2f} L/h",
            f"{k_elim:.4f} 1/h",
            f"{half_life_h:.2f} h",
            f"{metrics['Cmax_last_interval']:.2f} mg/L",
            f"{metrics['Tmax_after_dose']:.2f} h",
            f"{metrics['Cmin_last_interval']:.2f} mg/L",
            f"{metrics['AUC24']:.2f} mg*h/L",
            f"{mic_mg_l:.2f} mg/L",
            f"{cmax_mic:.2f}",
            status_label(cmax_mic, target_cmax_mic, direction="above"),
            status_label(metrics['Cmin_last_interval'], trough_threshold, direction="below")
        ]
    })

    display(summary)


interact(
    plot_aminoglycoside_multiple_dosing,
    body_weight_kg=FloatSlider(value=70, min=40, max=120, step=5, description="Weight"),
    dose_mg_kg=FloatSlider(value=5.0, min=1.0, max=8.0, step=0.5, description="Dose mg/kg"),
    tau_h=FloatSlider(value=24, min=6, max=48, step=6, description="Tau"),
    infusion_h=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.5, description="Infusion"),
    n_doses=IntSlider(value=5, min=1, max=10, step=1, description="Doses"),
    vd_l_kg=FloatSlider(value=0.25, min=0.15, max=0.50, step=0.05, description="Vd L/kg"),
    crcl_ml_min=FloatSlider(value=100, min=15, max=150, step=5, description="CrCl"),
    mic_mg_l=FloatSlider(value=1.0, min=0.25, max=8.0, step=0.25, description="MIC"),
    target_cmax_mic=FloatSlider(value=8, min=4, max=12, step=1, description="Target"),
    trough_threshold=FloatSlider(value=2.0, min=0.5, max=5.0, step=0.5, description="Trough")
);

interactive(children=(FloatSlider(value=70.0, description='Weight', max=120.0, min=40.0, step=5.0), FloatSlide…

## 6. Observation Task 1: Cmax/MIC and the Efficacy Target

Complete the following tasks.

### Task A: Standard Extended-Interval Dosing

Set:

- Weight = 70 kg
- Dose = 5 mg/kg
- Tau = 24 h
- Infusion = 1 h
- Vd = 0.25 L/kg
- CrCl = 100 mL/min
- MIC = 1 mg/L
- Target = 8

Observe:

- What is Cmax?
- Does Cmax/MIC reach the target?
- Is Cmin relatively low?

### Task B: Increased MIC

Change only the MIC to 2 mg/L, and then to 4 mg/L.

Observe:

- Does Cmax change?
- Does Cmax/MIC decrease?
- Is it harder for the same dosing regimen to reach the target when the pathogen has a higher MIC?

### Task C: Increased Volume of Distribution

Change Vd from 0.25 L/kg to 0.40 L/kg.

Observe:

- Does Cmax decrease?
- Does Cmax/MIC decrease?
- What types of patients might this represent?

Hint: Severe infection, fluid resuscitation, burns, sepsis, and similar conditions may alter the volume of distribution.


## 7. Once-Daily Dosing Versus Traditional Divided Dosing

An important teaching point for aminoglycosides is:

> With the same total daily dose, divided dosing and once-daily dosing may produce different peak and trough concentrations.

A traditional dosing regimen may use smaller doses given more frequently, for example:

$$
1.7\ mg/kg\ q8h
$$

Extended-interval dosing may use a larger single dose with a longer interval, for example:

$$
5\ mg/kg\ q24h
$$

The total daily doses are similar, but the concentration-time profiles are different.

In general, you may observe that:

- Once-daily dosing produces a higher peak concentration, making Cmax/MIC more likely to reach the target.
- Once-daily dosing uses a longer dosing interval, so the trough concentration may be lower.
- Traditional divided dosing produces a lower peak concentration but smaller concentration fluctuations.

This reflects the difference between aminoglycosides and beta-lactam antibiotics:

- Beta-lactams mainly focus on %fT > MIC.
- Aminoglycosides mainly focus on Cmax/MIC while also considering trough concentration for safety.


In [ ]:
def compare_traditional_vs_extended_interval(
    body_weight_kg=70,
    vd_l_kg=0.25,
    crcl_ml_min=100,
    mic_mg_l=1.0,
    infusion_h=1.0,
    n_days=3,
    trough_threshold=2.0,
    target_cmax_mic=8
):
    regimens = {
        "Traditional: 1.7 mg/kg q8h": {"dose_mg_kg": 1.7, "tau_h": 8},
        "Extended interval: 5 mg/kg q24h": {"dose_mg_kg": 5.0, "tau_h": 24}
    }

    vd_l = body_weight_kg * vd_l_kg
    cl_l_h = gentamicin_clearance_from_crcl(crcl_ml_min)
    #t_end_h = 24 * n_days
    #t = np.linspace(0, t_end_h, 4000)

    # 确保模拟时间足够长以包含最后一次给药后的完整间隔
    t_end_h = 24 * n_days
    t = np.linspace(0, t_end_h + 24, 5000) # 额外增加24h确保覆盖最后一个tau

    fig, ax = plt.subplots(figsize=(10, 6))
    summary_rows = []

    for name, regimen in regimens.items():
        dose_mg = body_weight_kg * regimen["dose_mg_kg"]
        tau_h = regimen["tau_h"]
        # 计算在n_days内的给药总次数
        n_doses = int(t_end_h / tau_h)
        #n_doses = int(np.floor(t_end_h / tau_h)) + 1

        concentration, k_elim, half_life_h = one_compartment_iv_infusion_multiple(
            t=t,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            tau_h=tau_h,
            infusion_h=infusion_h,
            n_doses=n_doses
        )

        metrics = calculate_interval_metrics(
            t=t,
            concentration=concentration,
            tau_h=tau_h,
            infusion_h=infusion_h,
            n_doses=n_doses
        )

        cmax_mic = metrics["Cmax_last_interval"] / mic_mg_l

        #ax.plot(t, concentration, linewidth=2, label=name)
        # 只绘制用户请求的天数范围内的曲线
        plot_mask = t <= t_end_h
        ax.plot(t[plot_mask], concentration[plot_mask], linewidth=2, label=name)

        summary_rows.append({
            "Regimen": name,
            "Dose": f"{dose_mg:.1f} mg",
            "Tau": f"{tau_h:.0f} h",
            "Daily dose": f"{dose_mg * 24 / tau_h:.1f} mg/day",
            "Cmax": f"{metrics['Cmax_last_interval']:.2f} mg/L",
            "Cmin": f"{metrics['Cmin_last_interval']:.2f} mg/L",
            "Cmax/MIC": f"{cmax_mic:.2f}",
            "Efficacy target": status_label(cmax_mic, target_cmax_mic, "above"),
            "Trough risk": status_label(metrics['Cmin_last_interval'], trough_threshold, "below")
        })

    ax.axhline(mic_mg_l, linestyle="--", label=f"MIC = {mic_mg_l:.2f} mg/L")
    ax.axhline(trough_threshold, linestyle=":", label=f"Trough threshold = {trough_threshold:.1f} mg/L")
    ax.set_title("Traditional vs Extended-Interval Aminoglycoside Dosing")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    display(pd.DataFrame(summary_rows))


interact(
    compare_traditional_vs_extended_interval,
    body_weight_kg=FloatSlider(value=70, min=40, max=120, step=5, description="Weight"),
    vd_l_kg=FloatSlider(value=0.25, min=0.15, max=0.50, step=0.05, description="Vd L/kg"),
    crcl_ml_min=FloatSlider(value=100, min=15, max=150, step=5, description="CrCl"),
    mic_mg_l=FloatSlider(value=1.0, min=0.25, max=8.0, step=0.25, description="MIC"),
    infusion_h=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.5, description="Infusion"),
    n_days=IntSlider(value=3, min=1, max=7, step=1, description="Days"),
    trough_threshold=FloatSlider(value=2.0, min=0.5, max=5.0, step=0.5, description="Trough"),
    target_cmax_mic=FloatSlider(value=8, min=4, max=12, step=1, description="Target")
);

interactive(children=(FloatSlider(value=70.0, description='Weight', max=120.0, min=40.0, step=5.0), FloatSlide…

## 8. Observation Task 2: Comparing Two Dosing Approaches

Use the simulation above to complete the following tasks.

### Task A: Normal Renal Function

Set:

- Weight = 70 kg
- Vd = 0.25 L/kg
- CrCl = 100 mL/min
- MIC = 1 mg/L

Compare:

- Cmax/MIC with traditional divided dosing
- Cmax/MIC with once-daily dosing
- Cmin for both regimens

Think about:

- Which regimen is more likely to achieve a higher Cmax/MIC?
- Which regimen produces a lower concentration at the end of the dosing interval?

### Task B: Increased MIC

Change the MIC to 2 mg/L or 4 mg/L.

Observe:

- Which regimen is more likely to maintain Cmax/MIC target attainment?
- When MIC increases, why does PK/PD target attainment decrease even if the dose is unchanged?

### Task C: Reduced Renal Function

Change CrCl to 30 mL/min.

Observe:

- Is the half-life prolonged?
- Does the trough concentration increase?
- Does this suggest a need to extend the dosing interval or perform TDM?


## 9. Renal Function Changes and Aminoglycoside Accumulation

Gentamicin is mainly cleared by the kidneys. Therefore, changes in renal function can markedly affect systemic exposure.

In a one-compartment model:

$$
k = \frac{CL}{V_d}
$$

$$
t_{1/2} = \frac{0.693 \times V_d}{CL}
$$

When renal function declines, CL decreases, leading to:

- Lower k
- Prolonged half-life
- Slower concentration decline
- Higher trough concentration
- Increased accumulation and toxicity risk

This is an important reason why aminoglycoside dose and/or dosing interval should be adjusted according to renal function.


In [ ]:
def plot_renal_function_effect(
    body_weight_kg=70,
    dose_mg_kg=5.0,
    tau_h=24,
    infusion_h=1.0,
    vd_l_kg=0.25,
    mic_mg_l=1.0,
    n_days=5,
    trough_threshold=2.0
):
    crcl_values = [100, 60, 30, 15]
    vd_l = body_weight_kg * vd_l_kg
    dose_mg = body_weight_kg * dose_mg_kg
    t_end_h = 24 * n_days
    t = np.linspace(0, t_end_h, 5000)

    fig, ax = plt.subplots(figsize=(10, 6))
    summary_rows = []

    for crcl in crcl_values:
        cl_l_h = gentamicin_clearance_from_crcl(crcl)
        n_doses = int(np.floor(t_end_h / tau_h)) + 1

        concentration, k_elim, half_life_h = one_compartment_iv_infusion_multiple(
            t=t,
            dose_mg=dose_mg,
            vd_l=vd_l,
            cl_l_h=cl_l_h,
            tau_h=tau_h,
            infusion_h=infusion_h,
            n_doses=n_doses
        )

        metrics = calculate_interval_metrics(
            t=t,
            concentration=concentration,
            tau_h=tau_h,
            infusion_h=infusion_h,
            n_doses=n_doses
        )

        ax.plot(t, concentration, linewidth=2, label=f"CrCl {crcl} mL/min")

        summary_rows.append({
            "CrCl": f"{crcl} mL/min",
            "CL": f"{cl_l_h:.2f} L/h",
            "Half-life": f"{half_life_h:.2f} h",
            "Cmax": f"{metrics['Cmax_last_interval']:.2f} mg/L",
            "Cmin": f"{metrics['Cmin_last_interval']:.2f} mg/L",
            "AUC24": f"{metrics['AUC24']:.2f} mg*h/L"
        })

    ax.axhline(mic_mg_l, linestyle="--", label=f"MIC = {mic_mg_l:.2f} mg/L")
    ax.axhline(trough_threshold, linestyle=":", label=f"Trough threshold = {trough_threshold:.1f} mg/L")
    ax.set_title("Effect of Renal Function on Gentamicin Accumulation")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Concentration (mg/L)")
    ax.legend()
    plt.show()

    display(pd.DataFrame(summary_rows))


interact(
    plot_renal_function_effect,
    body_weight_kg=FloatSlider(value=70, min=40, max=120, step=5, description="Weight"),
    dose_mg_kg=FloatSlider(value=5.0, min=1.0, max=8.0, step=0.5, description="Dose mg/kg"),
    tau_h=FloatSlider(value=24, min=6, max=48, step=6, description="Tau"),
    infusion_h=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.5, description="Infusion"),
    vd_l_kg=FloatSlider(value=0.25, min=0.15, max=0.50, step=0.05, description="Vd L/kg"),
    mic_mg_l=FloatSlider(value=1.0, min=0.25, max=8.0, step=0.25, description="MIC"),
    n_days=IntSlider(value=5, min=2, max=10, step=1, description="Days"),
    trough_threshold=FloatSlider(value=2.0, min=0.5, max=5.0, step=0.5, description="Trough")
);

interactive(children=(FloatSlider(value=70.0, description='Weight', max=120.0, min=40.0, step=5.0), FloatSlide…

## 10. Observation Task 3: Impact of Reduced Renal Function

Use the simulation above to complete the following tasks.

### Task A: Compare Different CrCl Values

Keep the other parameters unchanged and observe the following levels of renal function:

- CrCl = 100 mL/min
- CrCl = 60 mL/min
- CrCl = 30 mL/min
- CrCl = 15 mL/min

Record:

- CL
- Half-life
- Cmax
- Cmin
- AUC24

### Task B: Assess Accumulation Risk

When CrCl decreases, observe:

- Does the concentration decline more slowly?
- Does Cmin increase?
- Is there obvious accumulation after multiple doses?

### Task C: Clinical Interpretation

Think about:

- Why should aminoglycosides be used cautiously in patients with reduced renal function?
- Why might the peak concentration after the first dose be insufficient for assessing safety?
- Why is monitoring trough concentration or extending the dosing interval clinically meaningful?


## 11. Effect of MIC on Cmax/MIC

PK/PD evaluation depends not only on drug concentration, but also on the pathogen MIC.

If Cmax remains unchanged:

$$
MIC \uparrow \Rightarrow C_{max}/MIC \downarrow
$$

In other words, even when a patient receives the same dose, Cmax/MIC may fail to reach the target if the pathogen MIC is higher.

The simulation below fixes the dosing regimen and observes how Cmax/MIC changes across different MIC values.


In [ ]:
def plot_cmax_mic_vs_mic(
    body_weight_kg=70,
    dose_mg_kg=5.0,
    tau_h=24,
    infusion_h=1.0,
    vd_l_kg=0.25,
    crcl_ml_min=100,
    n_doses=5,
    target_cmax_mic=8
):
    mic_values = np.array([0.25, 0.5, 1, 2, 4, 8])
    dose_mg = body_weight_kg * dose_mg_kg
    vd_l = body_weight_kg * vd_l_kg
    cl_l_h = gentamicin_clearance_from_crcl(crcl_ml_min)

    t_end_h = tau_h * n_doses
    t = np.linspace(0, t_end_h, 3000)

    concentration, k_elim, half_life_h = one_compartment_iv_infusion_multiple(
        t=t,
        dose_mg=dose_mg,
        vd_l=vd_l,
        cl_l_h=cl_l_h,
        tau_h=tau_h,
        infusion_h=infusion_h,
        n_doses=n_doses
    )

    metrics = calculate_interval_metrics(
        t=t,
        concentration=concentration,
        tau_h=tau_h,
        infusion_h=infusion_h,
        n_doses=n_doses
    )

    cmax = metrics["Cmax_last_interval"]
    ratios = cmax / mic_values

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(mic_values, ratios, marker="o", linewidth=2)
    ax.axhline(target_cmax_mic, linestyle="--", label=f"Target = {target_cmax_mic:.0f}")
    ax.set_xscale("log", base=2)
    ax.set_title("Effect of MIC on Cmax/MIC")
    ax.set_xlabel("MIC (mg/L)")
    ax.set_ylabel("Cmax/MIC")
    ax.legend()
    plt.show()

    summary = pd.DataFrame({
        "MIC (mg/L)": mic_values,
        "Cmax (mg/L)": [round(cmax, 2)] * len(mic_values),
        "Cmax/MIC": np.round(ratios, 2),
        "Status": [status_label(r, target_cmax_mic, "above") for r in ratios]
    })

    display(summary)


interact(
    plot_cmax_mic_vs_mic,
    body_weight_kg=FloatSlider(value=70, min=40, max=120, step=5, description="Weight"),
    dose_mg_kg=FloatSlider(value=5.0, min=1.0, max=8.0, step=0.5, description="Dose mg/kg"),
    tau_h=FloatSlider(value=24, min=6, max=48, step=6, description="Tau"),
    infusion_h=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.5, description="Infusion"),
    vd_l_kg=FloatSlider(value=0.25, min=0.15, max=0.50, step=0.05, description="Vd L/kg"),
    crcl_ml_min=FloatSlider(value=100, min=15, max=150, step=5, description="CrCl"),
    n_doses=IntSlider(value=5, min=1, max=10, step=1, description="Doses"),
    target_cmax_mic=FloatSlider(value=8, min=4, max=12, step=1, description="Target")
);

## 12. Observation Task 4: PK/PD Risk When MIC Increases

Use the simulation above to complete the following tasks.

### Task A: Standard Regimen

Set:

- Weight = 70 kg
- Dose = 5 mg/kg
- Tau = 24 h
- Vd = 0.25 L/kg
- CrCl = 100 mL/min
- Target = 8

Observe Cmax/MIC at different MIC values.

### Task B: Determine the MIC Range for Target Attainment

Record at which MIC values Cmax/MIC can reach the target.

Think about:

- What happens to Cmax/MIC when MIC increases from 1 mg/L to 2 mg/L?
- When MIC is high, is simply increasing the dose always feasible?
- Why does clinical interpretation of PK/PD require pathogen susceptibility results?


## 13. Self-Test Questions: Aminoglycoside PK/PD

Complete the following self-test questions based on the content of this notebook. It is recommended that you answer independently first and then check the reference answers in the next cell.

---

### Question 1: Which PK/PD index is most commonly used for aminoglycosides?

A. %fT > MIC  
B. AUC24/MIC  
C. Cmax/MIC  
D. Tmax/MIC  

---

### Question 2: If Cmax remains unchanged, what happens when MIC increases?

A. Cmax/MIC increases  
B. Cmax/MIC decreases  
C. Cmin always decreases  
D. Half-life is always shortened  

---

### Question 3: What is the most likely effect of reduced renal function on gentamicin?

A. Clearance increases and half-life shortens  
B. Clearance decreases and half-life is prolonged  
C. Volume of distribution necessarily becomes 0  
D. Cmax/MIC is unrelated to MIC  

---

### Question 4: Compared with traditional divided dosing, what is an important teaching feature of extended-interval dosing?

A. The single dose is smaller, so peak concentration must be lower  
B. The single dose is larger and may produce higher Cmax/MIC  
C. Renal function does not need to be considered  
D. Trough concentration does not need to be considered  

---

### Question 5: Why should trough concentration be considered for aminoglycosides?

A. Trough concentration mainly reflects absorption rate  
B. The higher the trough concentration, the better the efficacy must be  
C. An elevated trough concentration may indicate accumulation and toxicity risk  
D. Trough concentration is completely unrelated to safety


## 14. Reference Answers for the Self-Test Questions

### Question 1

**Reference answer: C**

**Explanation:**  
Aminoglycosides are concentration-dependent bactericidal drugs. In teaching settings, Cmax/MIC is commonly used to evaluate their antibacterial effect. A higher Cmax, especially a higher multiple relative to MIC, is generally more favorable for achieving the antibacterial target.

---

### Question 2

**Reference answer: B**

**Explanation:**  
When Cmax remains unchanged:

$$
C_{max}/MIC = \frac{C_{max}}{MIC}
$$

An increase in MIC makes the denominator larger, so Cmax/MIC decreases. This means that it becomes more difficult for the same dosing regimen to achieve PK/PD target attainment against a pathogen with a higher MIC.

---

### Question 3

**Reference answer: B**

**Explanation:**  
Gentamicin is mainly cleared by the kidneys. When renal function declines, clearance decreases, elimination becomes slower, half-life is prolonged, and trough concentration and accumulation risk may increase.

---

### Question 4

**Reference answer: B**

**Explanation:**  
Extended-interval dosing usually uses a larger single dose and a longer dosing interval, so it may produce a higher peak concentration and higher Cmax/MIC. The longer dosing interval also allows concentration to decline. However, renal function and trough concentration still need to be considered.

---

### Question 5

**Reference answer: C**

**Explanation:**  
The safety of aminoglycosides is closely related to accumulation. An elevated trough concentration may suggest insufficient drug clearance and increased accumulation, which may increase the risk of nephrotoxicity and ototoxicity. Therefore, TDM is often needed in clinical practice to assess whether the dose and dosing interval are appropriate.


## 15. Notebook Summary

This notebook used gentamicin as a teaching example to introduce aminoglycoside PK/PD simulation.

Key takeaways:

1. Aminoglycosides show concentration-dependent antibacterial activity, so peak concentration is closely related to efficacy.
2. Cmax/MIC is a key PK/PD index, and a target of approximately 8–10 is often used in teaching simulations.
3. Extended-interval dosing may improve peak concentration and Cmax/MIC compared with traditional divided dosing.
4. Elevated trough concentration may indicate accumulation and increased toxicity risk.
5. Renal function strongly affects gentamicin clearance, half-life, trough concentration, and accumulation.
6. Simplified simulations help explain dosing logic, but they cannot replace TDM or clinical judgment.

The complete logic of this section can be summarized as:

$$
Dose\ regimen \rightarrow C_{max} \rightarrow C_{max}/MIC \rightarrow Efficacy
$$

At the same time, safety should be considered through:

$$
Renal\ function \rightarrow CL \rightarrow C_{min} \rightarrow Accumulation/Toxicity\ risk
$$

## 16. References

The specific drug example and teaching targets in this notebook are mainly based on the following references:

1. DailyMed. Gentamicin Sulfate in Sodium Chloride Injection, Clinical Pharmacology and Dosage and Administration.  
   https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=449c9a41-d61e-49fd-b5e3-d76f74b92acf

2. DailyMed. Gentamicin Injection, USP.  
   https://dailymed.nlm.nih.gov/dailymed/fda/fdaDrugXsl.cfm?setid=a73a5453-c091-43fd-aae2-d992152363b1

3. Kashuba ADM, Nafziger AN, Drusano GL, Bertino JS. Optimizing aminoglycoside therapy for nosocomial pneumonia caused by gram-negative bacteria. Antimicrob Agents Chemother. 1999;43(3):623-629.  
   https://pmc.ncbi.nlm.nih.gov/articles/PMC105693/

4. Krause KM, Serio AW, Kane TR, Connolly LE. Aminoglycosides: An Overview. Cold Spring Harb Perspect Med. 2016;6(6):a027029.  
   https://pmc.ncbi.nlm.nih.gov/articles/PMC4888811/

Note: The models, parameters, and thresholds in this notebook are simplified for teaching purposes and are intended to help learners understand PK/PD reasoning. They should not be used for real clinical prescribing.
